# Validaciones de mediciones Bronze

Notebook de solo lectura. Ejecutarlo después de cada carga y antes y después de repetir la ingesta sin archivos nuevos. El conteo del segundo control debe ser igual al primero.


In [3]:
import json
from pathlib import Path
from pyspark.sql import functions as F
from pyspark.sql.types import StringType

config_path = next((parent / 'config' / 'bronze.json' for parent in (Path.cwd(), *Path.cwd().parents) if (parent / 'config' / 'bronze.json').is_file()), None)
if config_path is None:
    raise FileNotFoundError('No se encuentra config/bronze.json')
config = json.loads(config_path.read_text(encoding='utf-8'))
INPUT_DIR = config['mediciones_input_dir']
TARGET_TABLE = config['tables']['mediciones']
raw_columns = list(config['input_schemas']['mediciones'])
df = spark.table(TARGET_TABLE)
assert all(df.schema[name].dataType == StringType() for name in raw_columns)
assert '_source_file' in df.columns
assert df.filter(F.col('_source_file').isNull()).count() == 0
assert df.filter(~F.col('_source_file').startswith(INPUT_DIR + '/')).count() == 0
total = df.count()
print('Total Bronze:', total)
display(df.groupBy('_source_file').count().orderBy('_source_file'))


Total Bronze: 473


,_source_file,count
0,/Volumes/workspace/lensdata_bronze/archivos/recepcion/mediciones/lote_01/mediciones_2026-11-01.csv,40
1,/Volumes/workspace/lensdata_bronze/archivos/recepcion/mediciones/lote_02/mediciones_2026-11-02.csv,40
2,/Volumes/workspace/lensdata_bronze/archivos/recepcion/mediciones/lote_03/mediciones_2026-11-03.csv,39
3,/Volumes/workspace/lensdata_bronze/archivos/recepcion/mediciones/lote_03/mediciones_2026-11-04.csv,39
4,/Volumes/workspace/lensdata_bronze/archivos/recepcion/mediciones/lote_03/mediciones_2026-11-05.csv,39
5,/Volumes/workspace/lensdata_bronze/archivos/recepcion/mediciones/lote_03/mediciones_2026-11-06.csv,39
6,/Volumes/workspace/lensdata_bronze/archivos/recepcion/mediciones/lote_03/mediciones_2026-11-07.csv,39
7,/Volumes/workspace/lensdata_bronze/archivos/recepcion/mediciones/lote_03/mediciones_2026-11-08.csv,39
8,/Volumes/workspace/lensdata_bronze/archivos/recepcion/mediciones/lote_03/mediciones_2026-11-09.csv,39
9,/Volumes/workspace/lensdata_bronze/archivos/recepcion/mediciones/lote_03/mediciones_2026-11-10.csv,39


## Formato y conservación de registros

Las consultas siguientes no modifican la tabla. Las comprobaciones de reenvío y corrección se activan cuando ambos CSV están cargados.


In [4]:
detail = spark.sql(f'DESCRIBE DETAIL {TARGET_TABLE}').select('format').first()
assert detail['format'].lower() == 'delta'
print('Formato:', detail['format'])

unknown_path = INPUT_DIR + '/lote_03/video_desconocido.csv'
unknown = df.filter((F.col('_source_file') == unknown_path) & (F.col('video_id') == 'V999'))
print('Vídeo desconocido conservado:', unknown.count())
display(unknown.select(*raw_columns, '_source_file'))

original = df.filter((F.col('plataforma') == 'tiktok') & (F.col('video_id') == 'V000') & (F.col('fecha_medicion') == '2026-11-01'))
display(original.select(*raw_columns, '_source_file').orderBy('_source_file'))


Formato: delta
Vídeo desconocido conservado: 1


,plataforma,video_id,fecha_medicion,version_origen,reproducciones,me_gusta,comentarios,compartidos,_source_file
0,tiktok,V999,2026-11-03,1,100,5,1,1,/Volumes/workspace/lensdata_bronze/archivos/recepcion/mediciones/lote_03/video_desconocido.csv


,plataforma,video_id,fecha_medicion,version_origen,reproducciones,me_gusta,comentarios,compartidos,_source_file
0,tiktok,V000,2026-11-01,1,1000,50,14,11,/Volumes/workspace/lensdata_bronze/archivos/recepcion/mediciones/lote_01/mediciones_2026-11-01.csv
1,tiktok,V000,2026-11-01,2,950,47,13,10,/Volumes/workspace/lensdata_bronze/archivos/recepcion/mediciones/lote_correccion/correccion_dia_1.csv
2,tiktok,V000,2026-11-01,1,1000,50,14,11,/Volumes/workspace/lensdata_bronze/archivos/recepcion/mediciones/lote_reenvio/reenvio_dia_1_v1.csv


## Prueba final tras reenvío y corrección

Ejecutar esta celda solo después de copiar los dos CSV a sus respectivas subcarpetas y volver a ejecutar la ingesta.


In [5]:
paths = {
    'original': INPUT_DIR + '/lote_01/mediciones_2026-11-01.csv',
    'reenvio': INPUT_DIR + '/lote_reenvio/reenvio_dia_1_v1.csv',
    'correccion': INPUT_DIR + '/lote_correccion/correccion_dia_1.csv',
    'desconocido': INPUT_DIR + '/lote_03/video_desconocido.csv',
}
for name, path in paths.items():
    count = df.filter(F.col('_source_file') == path).count()
    print(name, count)
    expected = 40 if name == 'original' else 1
    assert count == expected, f'Conteo inesperado: {name}={count}; esperado: {expected}'
assert df.filter((F.col('_source_file') == paths['reenvio']) & (F.col('version_origen') == '1') & (F.col('reproducciones') == '1000')).count() == 1
assert df.filter((F.col('_source_file') == paths['correccion']) & (F.col('version_origen') == '2') & (F.col('reproducciones') == '950')).count() == 1
assert df.filter((F.col('_source_file') == paths['desconocido']) & (F.col('video_id') == 'V999')).count() == 1
assert df.filter((F.col('plataforma') == 'tiktok') & (F.col('video_id') == 'V000') & (F.col('fecha_medicion') == '2026-11-01')).count() == 3
assert df.count() == 473, f'Total final inesperado: {df.count()}; esperado: 473'
assert df.select('_source_file').distinct().count() == 15
assert df.filter((F.col('_source_file') == INPUT_DIR + '/lote_01/mediciones_2026-11-01.csv') & (F.col('video_id') == 'V001') & (F.col('plataforma') == 'tiktok') & F.col('comentarios').isNull()).count() == 1
assert df.filter((F.col('_source_file') == INPUT_DIR + '/lote_02/mediciones_2026-11-02.csv') & (F.col('video_id') == 'V001') & (F.col('plataforma') == 'tiktok') & (F.col('comentarios') == 'pendiente')).count() == 1
assert df.filter((F.col('_source_file') == INPUT_DIR + '/lote_02/mediciones_2026-11-02.csv') & (F.col('video_id') == 'V002') & (F.col('plataforma') == 'instagram') & (F.col('compartidos') == '-1')).count() == 1
print('Bronze conserva el original, el reenvío y la corrección por separado.')


original 40
reenvio 1
correccion 1
desconocido 1
Bronze conserva el original, el reenvío y la corrección por separado.
